In [11]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

BASE_PATH = "/home/ayush/time_series_llm/models/base_model"
ADAPTER_PATH = "/home/ayush/time_series_llm/models/Mod_ver_1_syn_1/checkpoint-225"

# =========================
# TOKENIZER
# =========================
tokenizer = AutoTokenizer.from_pretrained(
    BASE_PATH,
    trust_remote_code=True
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# =========================
# BASE MODEL (FORCE GPU + SAFE LOAD)
# =========================
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_PATH,
    device_map={"": 0},          # 🔥 FORCE SINGLE GPU
    torch_dtype=torch.float16,
    trust_remote_code=True,
    low_cpu_mem_usage=True,
    offload_folder=None          # 🔥 CRITICAL: disables disk offload
)

# =========================
# LOAD LORA ADAPTER
# =========================
model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_PATH,
    is_trainable=False
)

model.eval()

print("✅ Model loaded on GPU only (no disk offload)")

# =========================
# GENERATION FUNCTION
# =========================
def generate(prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=120,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            eos_token_id=tokenizer.eos_token_id
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

Loading checkpoint shards: 100%|██████████| 2/2 [00:21<00:00, 10.91s/it]


✅ Model loaded on GPU only (no disk offload)


In [12]:
from transformers import AutoConfig

config = AutoConfig.from_pretrained("/home/ayush/time_series_llm/models/base_model")
print(config.model_type)
print(config.architectures)

qwen2
['Qwen2ForCausalLM']


In [13]:
prompt = """
You are a time series expert.

Explain ARIMA model in simple terms:
- What does ARIMA stand for?
- What does each component do (AR, I, MA)?
- When is it used?
- Give a simple example.

Be concise but technical.
"""

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        eos_token_id=tokenizer.eos_token_id
    )

print(tokenizer.decode(output[0], skip_special_tokens=True))


You are a time series expert.

Explain ARIMA model in simple terms:
- What does ARIMA stand for?
- What does each component do (AR, I, MA)?
- When is it used?
- Give a simple example.

Be concise but technical.
Sure! Here’s a brief explanation of the ARIMA model:

### What Does ARIMA Stand For?

**ARIMA** stands for **AutoRegressive Integrated Moving Average**. It's a class of models that can be used to forecast future values based on past data and past forecast errors.

### Components Explained

- **AR (AutoRegressive)**: Uses past values of the series itself as predictors.
- **I (Integrated)**: Removes trends and seasonality by differencing the data (subtracting previous period’s value).
- **MA (Moving Average)**: Uses past forecast errors to predict future values.

### When Is It Used?

ARIMA models are suitable when your time series has a clear trend, seasonal patterns, or both. They're widely applied in fields like economics, finance, and weather forecasting.

### Simple Example


In [14]:
prompt = """
You are a time series forecasting model.

Given the sequence:
[5, 10, 15, 20, 25]

Tasks:
1. Identify the pattern
2. Predict next 5 values
3. Explain the rule briefly

Be numeric and consistent. No long theory.
"""

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=120,
        do_sample=False,   # important for deterministic evaluation
        use_cache=True,
        eos_token_id=tokenizer.eos_token_id
    )

print(tokenizer.decode(output[0], skip_special_tokens=True))

/home/ayush/time_series_llm/llm_env/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:567: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/home/ayush/time_series_llm/llm_env/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:572: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/home/ayush/time_series_llm/llm_env/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:589: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(



You are a time series forecasting model.

Given the sequence:
[5, 10, 15, 20, 25]

Tasks:
1. Identify the pattern
2. Predict next 5 values
3. Explain the rule briefly

Be numeric and consistent. No long theory.
Sure, let's break down the task step by step:

### Step 1: Identify the Pattern
The given sequence is [5, 10, 15, 20, 25]. Observing this, we can see that each value increases by 5 from the previous one. This indicates an arithmetic progression with a common difference of 5.

### Step 2: Predict Next 5 Values
Given the pattern identified, we can predict the next 5 values in the sequence by adding 5 to each subsequent number.

- Current sequence: [5, 
